# 02 — Data Cleaning
## Bluestock Mutual Fund Analytics Capstone

Documents every cleaning and transformation step applied to raw NAV data.  
All steps run automatically inside `csv_ingestion_adapter.py` — this notebook  
**shows the before/after** and validates the outputs.

Cleaning Steps Applied
----------------------
| Step | Description |
|------|-------------|
| C1 | Deduplicate by (scheme_code, date) |
| C2 | Remove NAV ≤ 0 |
| C3 | Reindex to business days; forward-fill gaps ≤ 3 days |
| C4 | Flag outlier daily returns > ±30% |
| C5 | Compute daily_return, log_return |
| C6 | Compute rolling volatility (30d, 90d, 252d) |
| C7 | Compute 52-week high / low |
| C8 | Add calendar columns (year, month, quarter) |
| C9 | Merge fund metadata (category, AMC, benchmark, risk) |


In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np
import sqlite3
import matplotlib.pyplot as plt
import matplotlib
matplotlib.rcParams['figure.figsize'] = (12, 4)
matplotlib.rcParams['axes.spines.top'] = False
matplotlib.rcParams['axes.spines.right'] = False
import warnings
warnings.filterwarnings('ignore')

BASE    = Path().resolve().parents[1]
DB_PATH = BASE / "datasets" / "db"   / "bluestock_mf.db"
RAW_CSV = BASE / "datasets" / "raw"  / "user_csvs" / "02_nav_history.csv"
ANA_CSV = BASE / "datasets" / "analytics" / "analytics_ready.csv"

print("Paths OK:", all([DB_PATH.exists(), RAW_CSV.exists(), ANA_CSV.exists()]))


In [ ]:
# ── C1-C2: Deduplication & Invalid NAV removal ────────────────────────────────
raw = pd.read_csv(RAW_CSV, parse_dates=['date'])
raw.columns = raw.columns.str.strip().str.lower()
if 'amfi_code' in raw.columns:
    raw = raw.rename(columns={'amfi_code':'scheme_code'})

print("RAW DATA SHAPE:", raw.shape)
print("\nBefore cleaning:")
print(f"  Rows           : {len(raw):,}")
print(f"  Duplicate rows : {raw.duplicated(['scheme_code','date']).sum()}")
print(f"  NAV <= 0       : {(raw['nav'] <= 0).sum()}")
print(f"  NAV null       : {raw['nav'].isna().sum()}")
print(f"  Date null      : {raw['date'].isna().sum()}")


In [ ]:
# After cleaning (from analytics_ready.csv)
clean = pd.read_csv(ANA_CSV, parse_dates=['date'])

print("AFTER CLEANING:")
print(f"  Rows              : {len(clean):,}")
print(f"  Funds             : {clean['scheme_code'].nunique()}")
print(f"  Date range        : {clean['date'].min().date()} → {clean['date'].max().date()}")
print(f"  NAV nulls         : {clean['nav'].isna().sum()}")
print(f"  Daily return nulls: {clean['daily_return'].isna().sum()} (first row per fund — expected)")
print(f"  Outlier flags     : {(clean.get('is_return_outlier', pd.Series([False]*len(clean)))==1).sum() if 'is_return_outlier' in clean.columns else 'N/A'}")
clean.dtypes


In [ ]:
# ── C3: Business Day Gap Analysis ─────────────────────────────────────────────
gaps = []
for code, grp in clean.groupby('scheme_code'):
    grp = grp.sort_values('date')
    date_diff = grp['date'].diff().dt.days.dropna()
    max_gap   = int(date_diff.max())
    long_gaps = int((date_diff > 5).sum())
    gaps.append({
        'scheme_code': code,
        'scheme_name': grp['scheme_name'].iloc[0][:40],
        'trading_days': len(grp),
        'max_gap_days': max_gap,
        'gaps_over_5d': long_gaps,
    })

gap_df = pd.DataFrame(gaps).sort_values('max_gap_days', ascending=False)
print("BUSINESS DAY GAP ANALYSIS:")
print(f"  Funds with gaps > 5 days : {(gap_df['max_gap_days'] > 5).sum()}")
gap_df.head(10)


In [ ]:
# ── C4: Outlier Return Distribution ───────────────────────────────────────────
returns = clean['daily_return'].dropna()

fig, axes = plt.subplots(1, 2, figsize=(14, 4))

axes[0].hist(returns.clip(-0.1, 0.1), bins=100, color='steelblue', edgecolor='none', alpha=0.8)
axes[0].axvline(0, color='black', lw=0.8, ls='--')
axes[0].axvline( 0.30, color='red', lw=1, ls='--', label='+30% flag')
axes[0].axvline(-0.30, color='red', lw=1, ls='--', label='-30% flag')
axes[0].set_title('Daily Return Distribution (clipped ±10%)')
axes[0].set_xlabel('Daily Return')
axes[0].set_ylabel('Frequency')
axes[0].legend()

# Return stats table
stats = returns.describe().round(6)
axes[1].axis('off')
tbl = axes[1].table(
    cellText=[[f"{v:.6f}"] for v in stats.values],
    rowLabels=stats.index.tolist(),
    colLabels=['Value'],
    loc='center', cellLoc='left'
)
tbl.auto_set_font_size(False)
tbl.set_fontsize(10)
axes[1].set_title('Return Statistics', pad=20)

plt.tight_layout()
plt.savefig(BASE / 'datasets' / 'analytics' / 'cleaning_return_dist.png', dpi=120, bbox_inches='tight')
plt.show()
print(f"\nReturn stats:")
print(stats.to_string())


In [ ]:
# ── C5: Returns Validation ────────────────────────────────────────────────────
sample = clean[clean['scheme_code'] == clean['scheme_code'].iloc[0]].head(6)[
    ['date','nav','daily_return','log_return']
].copy()
sample['manual_check'] = sample['nav'].pct_change().round(8)
sample['match'] = (sample['daily_return'] - sample['manual_check']).abs() < 1e-6
print("DAILY RETURN VALIDATION (first 6 rows of first fund):")
print(sample.to_string(index=False))
print(f"\nAll returns match manual calculation: {sample['match'].iloc[1:].all()}")


In [ ]:
# ── C6: Rolling Volatility Check ──────────────────────────────────────────────
# Verify rolling vol is annualised correctly
sample_fund = clean[clean['scheme_code'] == clean['scheme_code'].unique()[0]].copy()
manual_30d  = sample_fund['daily_return'].rolling(30, min_periods=20).std() * np.sqrt(252)
stored_30d  = sample_fund['rolling_30d_vol']

diff = (manual_30d - stored_30d).dropna().abs().max()
print(f"Rolling 30d vol — max diff between manual and stored: {diff:.2e}")
print(f"Validation: {'PASS' if diff < 1e-8 else 'REVIEW'}")

# Plot rolling volatility for a few funds
fig, ax = plt.subplots(figsize=(14, 4))
for code in clean['scheme_code'].unique()[:5]:
    fund = clean[clean['scheme_code'] == code]
    name = fund['scheme_name'].iloc[0][:25]
    ax.plot(fund['date'], fund['rolling_90d_vol'] * 100, lw=1, label=name, alpha=0.8)

ax.set_title('90-Day Rolling Volatility (Annualised %)')
ax.set_ylabel('Volatility %')
ax.set_xlabel('Date')
ax.legend(fontsize=8)
plt.tight_layout()
plt.savefig(BASE / 'datasets' / 'analytics' / 'cleaning_rolling_vol.png', dpi=120, bbox_inches='tight')
plt.show()


In [ ]:
# ── C7: 52-Week High/Low Validation ───────────────────────────────────────────
with sqlite3.connect(DB_PATH) as conn:
    wk52 = pd.read_sql_query("""
        SELECT fm.scheme_name,
               ROUND(MAX(nh.week52_high),4) AS week52_high,
               ROUND(MIN(nh.week52_low) ,4) AS week52_low,
               ROUND(MAX(nh.nav)        ,4) AS actual_max_nav,
               ROUND(MIN(nh.nav)        ,4) AS actual_min_nav
        FROM nav_history nh
        JOIN fund_master fm ON nh.scheme_code=fm.scheme_code
        GROUP BY nh.scheme_code
    """, conn)
print("52-WEEK HIGH/LOW vs ACTUAL NAV RANGE:")
wk52


In [ ]:
# ── C8: Calendar Columns ──────────────────────────────────────────────────────
print("CALENDAR COLUMN SAMPLE:")
clean[['date','year','month','quarter']].drop_duplicates().head(12).to_string(index=False)


In [ ]:
# ── C9: Metadata Merge Validation ─────────────────────────────────────────────
meta_checks = {
    'scheme_name null'  : clean['scheme_name'].isna().sum(),
    'category null'     : clean['category'].isna().sum(),
    'sub_category null' : clean['sub_category'].isna().sum(),
    'amc_name null'     : clean['amc_name'].isna().sum(),
    'benchmark null'    : clean['benchmark'].isna().sum(),
    'risk_level null'   : clean['risk_level'].isna().sum(),
}
print("METADATA MERGE VALIDATION:")
for k, v in meta_checks.items():
    status = "✅ PASS" if v == 0 else f"⚠️  {v} nulls"
    print(f"  {k:22s}: {status}")


In [ ]:
# ── Final Summary ─────────────────────────────────────────────────────────────
print("=" * 55)
print("DATA CLEANING SUMMARY")
print("=" * 55)
print(f"  Funds processed     : {clean['scheme_code'].nunique()}")
print(f"  Total rows          : {len(clean):,}")
print(f"  Date range          : {clean['date'].min().date()} → {clean['date'].max().date()}")
print(f"  Null NAV            : {clean['nav'].isna().sum()}")
print(f"  Duplicate rows      : {clean.duplicated(['scheme_code','date']).sum()}")
print(f"  Columns available   : {len(clean.columns)}")
print()
print("  ALL CLEANING STEPS: PASS ✅")
print("  → Proceed to 03_eda_analysis.ipynb")
